In [ ]:
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
x = load_dotenv()

from typesafe_sdk import Choice, Noul, Score, TypeSafeClient
client = TypeSafeClient()

# Task 1

In [2]:
statement = "WKT(A): Point(-89.3551 43.123) WKT(B): Polygon (-89.3552 43.124, -89.355 43.124, -89.355 43.122, -89.3552  43.122, -89.3552 43.124)"

response = client.system_one(
    state=statement,
    questions={
        "Task3": Choice(
            instructions="You are given two geometries in the format WKT. Classify their relation according to the predicates defined by DE-9IM and the OGC.",
            criteria={
                "contain": "",
                "within": "",
                "touches": "",
                "crosses": "",
                "disjoint": "",
                "overlaps": "",
                "equals": "",
            },
        )
    },
)

print(response.answers["Task3"].choice)

KeyboardInterrupt: 

# Task 2

Not possible with a system one model, because a new geometry has to be *generated*

# Task 3 — Conversion of vernacular relation descriptions

model used: ModelMetadata(name='jev-latest', description="The latest iteration of TypeSafe's System One Model: Jev", release_date='2026-09-10T18:38:01.391457+00:00')

In [ ]:
# Data processing
pd.set_option('display.max_colwidth', None)
df_og = pd.read_csv('data/triplet_update_v3.csv', encoding='utf-8')


In [109]:
df_og[df_og['relation_predicate'] == 'abuts']['spatial_relation']

0    touches
Name: spatial_relation, dtype: str

In [130]:
df = df_og.drop(columns=['placetype_subject','geometry_type_subject','placetype_object','geometry_type_object','Sentence','place_name_subject','place_name_object','vernacular_relation'])
df['relation_predicate'] = df['relation_predicate'].drop_duplicates()
df = df[df['relation_predicate'].notna()]
df = df.reset_index(drop=True)
df


,spatial_relation,relation_predicate
0,touches,abuts
1,crosses,across
2,touches,is across from
3,disjoint,is adjacent to
4,touches,adjoins
...,...,...
98,contains,the site of
99,within,tip of
100,disjoint,touching
101,crosses,is upstream from


In [ ]:
for i in tqdm(range(len(df)), desc='Prompting'):
    statement = f"Place A {df.iloc[i]["relation_predicate"]} Place B."

    response = client.system_one(
        state=statement,
        questions={
            "Task3": Choice(
                instructions="To what topological predicate does the statement best map to? The predicates follow the definition of DE-9IM.",
                criteria={
                    "contains": "",
                    "within": "",
                    "touches": "",
                    "crosses": "",
                    "disjoint": "",
                    "overlaps": "",
                    "equals": "",
                },
            )
        },
    )

    df.loc[i, 'choice'] = response.answers['Task3'].choice
    df.loc[i, 'probabilities'] = str(response.answers['Task3'].probabilities)
    df.loc[i, 'confidence'] = response.answers['Task3'].confidence
    df.loc[i, 'tokens'] = str(response.usage)

Prompting: 100%|██████████| 103/103 [00:51<00:00,  1.98it/s]


In [134]:
df['Correct'] = (df['spatial_relation'] == df['choice']).astype(int)
print(df['Correct'].mean()*100)
df

32.038834951456316


,spatial_relation,relation_predicate,choice,probabilities,confidence,tokens,Correct
0,touches,abuts,touches,"{'contains': 0.0, 'within': 0.0, 'touches': 0.99, 'crosses': 0.0, 'disjoint': 0.0, 'overlaps': 0.01, 'equals': 0.0}",0.99,input_tokens=350 output_tokens=76,1
1,crosses,across,crosses,"{'contains': 0.0, 'within': 0.0, 'touches': 0.02, 'crosses': 0.94, 'disjoint': 0.01, 'overlaps': 0.03, 'equals': 0.0}",0.93,input_tokens=349 output_tokens=76,1
2,touches,is across from,disjoint,"{'contains': 0.0, 'within': 0.0, 'touches': 0.02, 'crosses': 0.02, 'disjoint': 0.96, 'overlaps': 0.0, 'equals': 0.0}",0.95,input_tokens=351 output_tokens=76,0
3,disjoint,is adjacent to,touches,"{'contains': 0.0, 'within': 0.0, 'touches': 0.99, 'crosses': 0.0, 'disjoint': 0.0, 'overlaps': 0.01, 'equals': 0.0}",0.99,input_tokens=351 output_tokens=76,0
4,touches,adjoins,touches,"{'contains': 0.0, 'within': 0.0, 'touches': 0.99, 'crosses': 0.0, 'disjoint': 0.0, 'overlaps': 0.01, 'equals': 0.0}",0.99,input_tokens=351 output_tokens=76,1
...,...,...,...,...,...,...,...
98,contains,the site of,within,"{'contains': 0.42, 'within': 0.43, 'touches': 0.1, 'crosses': 0.0, 'disjoint': 0.01, 'overlaps': 0.01, 'equals': 0.03}",0.33,input_tokens=351 output_tokens=74,0
99,within,tip of,touches,"{'contains': 0.09, 'within': 0.17, 'touches': 0.58, 'crosses': 0.02, 'disjoint': 0.09, 'overlaps': 0.03, 'equals': 0.02}",0.51,input_tokens=350 output_tokens=76,0
100,disjoint,touching,touches,"{'contains': 0.0, 'within': 0.0, 'touches': 1.0, 'crosses': 0.0, 'disjoint': 0.0, 'overlaps': 0.0, 'equals': 0.0}",1.00,input_tokens=349 output_tokens=76,0
101,crosses,is upstream from,disjoint,"{'contains': 0.0, 'within': 0.02, 'touches': 0.0, 'crosses': 0.01, 'disjoint': 0.97, 'overlaps': 0.0, 'equals': 0.0}",0.96,input_tokens=351 output_tokens=76,0


In [136]:
df.to_csv('results/run1.csv', index=False)

In [6]:
models = TypeSafeClient().models

In [8]:
modelsList = models.list()

In [20]:
data = modelsList.model_dump()
jev_latest = next((m for m in data['models'] if m['name'] == 'jev-latest'), None)
print(jev_latest)

{'name': 'jev-latest', 'description': "The latest iteration of TypeSafe's System One Model: Jev", 'release_date': '2026-09-10T18:38:01.391457+00:00'}
